# PAPC: Progressive Adaptive Predictive Coding for Pretrained ViTs
## CAISc 2026 — One Notebook, Four Sessions (A100 40GB, 9h each)

Set `SESSION` to `'A'`, `'B'`, `'C'`, or `'D'` and run the whole notebook.

| Session | What it produces | Est. time |
|---|---|---|
| **A** | Main SOTA table: 8 MedMNIST datasets × {Vanilla-B, PAPC-B, HP-ViT-B(w=0.05)} × 3 seeds | ~7h |
| **B** | Full ablation: 4 datasets × 6 conditions × 3 seeds + gate-clamping mechanism | ~6h |
| **C** | Cross-domain (CIFAR-100 at 3 sizes) + weight scaling-law sweep | ~6h |
| **D** | Extended 5-seed runs for tight CIs + per-layer weight visualization | ~7h |

**Novel contribution — PAPC:** cosine-warmup the auxiliary loss from 0 to per-layer learned weights. Early training = pure vanilla (features adapt to downstream task undisturbed). Late training = calibrated PC regularization (improves generalization). Combined with ViT-Base (IN21k→IN1k), this produces strong numbers.

## 0. Config

In [1]:
SESSION = 'A'   # <<< CHANGE THIS: 'A', 'B', 'C', or 'D'
NUM_SEEDS = 3
TIME_BUDGET_SEC = int(8.5 * 3600)  # 8.5h safe margin inside 9h session
print('SESSION =', SESSION)

SESSION = A


## 1. Setup

In [2]:
import subprocess, sys
subprocess.check_call([sys.executable,'-m','pip','install','-q',
    'medmnist','timm>=1.0','scikit-learn','pandas','matplotlib','scipy'])

import os, math, time, json, gc, warnings, traceback
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision, torchvision.transforms as T
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import roc_auc_score
from scipy import stats
import timm
from timm.data import Mixup
import medmnist
from medmnist import INFO

warnings.filterwarnings('ignore', category=UserWarning)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name} | {p.total_memory/1e9:.0f}GB')
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
HAS_BF16 = torch.cuda.is_bf16_supported() if device.type == 'cuda' else False
AMP_DTYPE = torch.bfloat16 if HAS_BF16 else torch.float16
torch.manual_seed(0); np.random.seed(0)

# ViT-Base IN21k->IN1k (strongest timm ViT-B, ~86M params)
MODEL_NAME = 'vit_base_patch16_224.augreg_in21k_ft_in1k'
IMG_SIZE = 224

for d in ('/teamspace/studios/this_studio','/kaggle/working','/content','.'):
    if os.path.isdir(d) and os.access(d, os.W_OK): OUTPUT_DIR = d; break
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output: {OUTPUT_DIR} | bf16: {HAS_BF16} | timm: {timm.__version__}')

GPU: NVIDIA A100-SXM4-40GB | 42GB
Output: /teamspace/studios/this_studio | bf16: True | timm: 1.0.27


## 2. Model — PAPC (Progressive Adaptive Predictive Coding)

In [3]:
# --- Vectorized diagonal SSM (causal conv1d form, math-equiv to loop) ---
class DiagonalSSM(nn.Module):
    def __init__(self, d_model, d_state=16, dt_min=1e-3, dt_max=1e-1):
        super().__init__()
        self.d_model, self.d_state = d_model, d_state
        a = torch.log(torch.linspace(1.0, float(d_state), d_state))
        self.A_log = nn.Parameter(a.repeat(d_model, 1))
        self.B = nn.Parameter(torch.randn(d_model, d_state) * 0.1)
        self.C = nn.Parameter(torch.randn(d_model, d_state) * 0.1)
        self.D = nn.Parameter(torch.ones(d_model))
        dt = torch.rand(d_model) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)
        self.dt_log = nn.Parameter(dt)

    def forward(self, x):
        B, L, D = x.shape
        dt = torch.exp(self.dt_log.float())
        A_bar = torch.exp(-torch.exp(self.A_log.float()) * dt.unsqueeze(-1))
        B_bar = self.B.float() * dt.unsqueeze(-1)
        logA = torch.log(A_bar.clamp(min=1e-38))
        k = torch.arange(L, device=x.device, dtype=torch.float32)
        A_pow = torch.exp(k.view(-1, 1, 1) * logA.unsqueeze(0))
        psi = (A_pow * (self.C.float() * B_bar).unsqueeze(0)).sum(-1).to(x.dtype)
        w = psi.t().contiguous().unsqueeze(1).flip(-1)
        xp = F.pad(x.transpose(1, 2).contiguous(), (L - 1, 0))
        h = F.conv1d(xp, w, groups=D)
        return (h + self.D.to(x.dtype).view(1, -1, 1) * x.transpose(1, 2)).transpose(1, 2)

# --- PC layer with gated error integration ---
class PCLayer(nn.Module):
    def __init__(self, dim, d_state=16, has_error=True):
        super().__init__()
        self.has_error = has_error
        self.pred_norm = nn.LayerNorm(dim, eps=1e-6)
        self.predictor = DiagonalSSM(dim, d_state=d_state)
        if has_error:
            self.err_norm = nn.LayerNorm(dim, eps=1e-6)
            self.err_proj = nn.Linear(dim, dim)
            self.gate = nn.Parameter(torch.zeros(1))

    def predict(self, x):
        return self.predictor(self.pred_norm(x))

    def integrate(self, actual, pred):
        return actual + torch.tanh(self.gate) * self.err_proj(self.err_norm(actual - pred))

    def gate_val(self):
        return 0.0 if not self.has_error else float(torch.tanh(self.gate).detach().cpu())

# --- Main model: vanilla / fixed-w / adaptive / progressive / PAPC ---
class PAPCViT(nn.Module):
    # mode flags: use_pc, adaptive, progressive, clamp_gate
    # PAPC = use_pc + adaptive + progressive (the novel method)
    def __init__(self, model_name=MODEL_NAME, img_size=IMG_SIZE, num_classes=10,
                 in_chans=3, d_state=16, pred_loss_weight=0.05, drop_path=0.1,
                 pretrained=True, use_pc=True, adaptive=False, progressive=False,
                 clamp_gate=False, w_max=0.01, w_l2=1e-3):
        super().__init__()
        self.use_pc = use_pc
        self.pred_loss_weight = pred_loss_weight if use_pc else 0.0
        self.clamp_gate = clamp_gate
        self.adaptive = adaptive and use_pc
        self.progressive = progressive
        self.w_max = w_max; self.w_l2 = w_l2
        self._step = 0; self._total = 1

        self.backbone = timm.create_model(model_name, pretrained=pretrained,
            img_size=img_size, num_classes=0, drop_path_rate=drop_path, in_chans=3)
        self.dim = self.backbone.embed_dim
        self.depth = len(self.backbone.blocks)
        self.in_chans = in_chans
        self.expand_gray = (in_chans == 1)

        if use_pc:
            self.pc = nn.ModuleList([
                PCLayer(self.dim, d_state, has_error=(i > 0))
                for i in range(self.depth)])
            if self.adaptive:
                # init so each weight starts near 0.002
                init_val = math.log(max(1e-6, math.exp(0.002 / max(1e-6, w_max)) - 1))
                self.log_w = nn.Parameter(torch.full((self.depth,), init_val))
        else:
            self.pc = None

        self.head = nn.Linear(self.dim, num_classes)
        nn.init.trunc_normal_(self.head.weight, std=0.02)
        nn.init.zeros_(self.head.bias)

    def set_progress(self, step, total):
        self._step = step; self._total = total

    def _prog_scale(self):
        # cosine warmup: 0 at start -> 1 at end
        if not self.progressive: return 1.0
        t = self._step / max(1, self._total)
        return 0.5 * (1.0 - math.cos(math.pi * t))

    def _layer_weights(self):
        if self.adaptive:
            return F.softplus(self.log_w) * self.w_max * self._prog_scale()
        return None

    def forward(self, x):
        if self.expand_gray: x = x.repeat(1, 3, 1, 1)
        x = self.backbone.patch_embed(x)
        x = self.backbone._pos_embed(x)
        if hasattr(self.backbone, 'patch_drop'): x = self.backbone.patch_drop(x)
        if hasattr(self.backbone, 'norm_pre'): x = self.backbone.norm_pre(x)

        pred_loss = x.new_zeros(())
        if self.use_pc:
            lw = self._layer_weights()
            prev = None
            for i, block in enumerate(self.backbone.blocks):
                actual = block(x); pc = self.pc[i]
                if prev is not None and pc.has_error:
                    wi = lw[i] if lw is not None else self._prog_scale()
                    pred_loss = pred_loss + wi * F.mse_loss(prev, actual.detach())
                    if not self.clamp_gate:
                        actual = pc.integrate(actual, prev)
                prev = pc.predict(actual); x = actual
            x = self.backbone.norm(x)
            if prev is not None:
                wl = lw[-1] if lw is not None else self._prog_scale()
                pred_loss = pred_loss + wl * F.mse_loss(prev, x.detach())
            if self.adaptive:
                pred_loss = pred_loss + self.w_l2 * (self._layer_weights() ** 2).sum()
            elif not self.adaptive:
                pred_loss = pred_loss * self.pred_loss_weight
        else:
            for block in self.backbone.blocks: x = block(x)
            x = self.backbone.norm(x)

        return self.head(x[:, 0]), pred_loss

    def gate_values(self):
        return [] if not self.use_pc else [pc.gate_val() for pc in self.pc]

    def learned_weights(self):
        if not self.adaptive or not self.use_pc: return None
        with torch.no_grad(): return (F.softplus(self.log_w) * self.w_max).cpu().tolist()

    def param_groups(self, base_lr, head_mult=10.0, pc_mult=5.0, decay=0.75):
        groups = []
        for i, block in enumerate(self.backbone.blocks):
            groups.append({'params': list(block.parameters()),
                           'lr': base_lr * decay ** (self.depth - 1 - i)})
        early = list(self.backbone.patch_embed.parameters())
        for attr in ('cls_token', 'pos_embed', 'reg_token'):
            pa = getattr(self.backbone, attr, None)
            if isinstance(pa, nn.Parameter): early.append(pa)
        groups.append({'params': early, 'lr': base_lr * decay ** self.depth})
        groups.append({'params': list(self.backbone.norm.parameters()), 'lr': base_lr})
        if self.use_pc:
            pp = list(self.pc.parameters())
            if self.adaptive: pp.append(self.log_w)
            groups.append({'params': pp, 'lr': base_lr * pc_mult})
        groups.append({'params': list(self.head.parameters()), 'lr': base_lr * head_mult})
        return [{'params': [p for p in g['params'] if isinstance(p, nn.Parameter)], 'lr': g['lr']}
                for g in groups
                if any(isinstance(p, nn.Parameter) for p in g['params'])]

# --- Quick smoke test ---
for tag, kw in [('vanilla', dict(use_pc=False)),
                ('fixed', dict(use_pc=True, pred_loss_weight=0.05)),
                ('PAPC', dict(use_pc=True, adaptive=True, progressive=True))]:
    m = PAPCViT(num_classes=8, in_chans=3, pretrained=False, **kw)
    n = sum(p.numel() for p in m.parameters())
    lo, pl = m(torch.randn(2, 3, 224, 224))
    print(f'  {tag:8s} params={n:>11,}  ploss={pl.item():.4f}')
    del m
print('Model OK')

  vanilla  params= 85,804,808  ploss=0.0000
  fixed    params= 92,797,459  ploss=0.1146
  PAPC     params= 92,797,471  ploss=0.0000
Model OK


## 3. Data + metrics

In [4]:
IM = ((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))

def build_tf(sz, ic, train):
    m, s = (IM[0], IM[1]) if ic == 3 else ((sum(IM[0]) / 3,), (sum(IM[1]) / 3,))
    if train:
        aug = T.RandAugment(2, 9) if ic == 3 else T.RandomAffine(10, (0.05, 0.05))
        return T.Compose([T.Resize(int(sz * 1.15), interpolation=3),
            T.RandomResizedCrop(sz, scale=(0.7, 1.0), interpolation=3),
            T.RandomHorizontalFlip(), aug, T.ToTensor(), T.Normalize(m, s),
            T.RandomErasing(p=0.25, scale=(0.02, 0.15))])
    return T.Compose([T.Resize(int(sz * 1.15), interpolation=3),
        T.CenterCrop(sz), T.ToTensor(), T.Normalize(m, s)])

class EMA:
    def __init__(self, model, decay=0.9995):
        self.decay = decay; self.sh = {k: v.detach().clone() for k, v in model.state_dict().items()}
    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point: self.sh[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else: self.sh[k] = v.detach().clone()
    def apply(self, model):
        bk = {k: v.detach().clone() for k, v in model.state_dict().items()}
        model.load_state_dict(self.sh, strict=True); return bk
    def restore(self, model, bk): model.load_state_dict(bk, strict=True)

def compute_metrics(yt, ys, task, nc):
    yt = np.asarray(yt); ys = np.nan_to_num(np.asarray(ys, dtype=np.float64))
    if task == 'multi-label, binary-class':
        yc = np.clip(ys, 0, 1); acc = float(((yc > 0.5).astype(int) == yt).mean())
        a = [roc_auc_score(yt[:, c], yc[:, c]) for c in range(nc) if 0 < yt[:, c].sum() < len(yt)]
        return acc, float(np.mean(a)) if a else float('nan')
    y1 = yt.squeeze().astype(np.int64)
    if ys.ndim > 1:
        rs = ys.sum(1, keepdims=True); yn = ys / np.where(rs > 0, rs, 1); yp = ys.argmax(1)
    else: yn = ys; yp = (ys > 0.5).astype(int)
    acc = float((yp == y1).mean())
    if task == 'binary-class' or nc == 2:
        sc = yn if yn.ndim == 1 else yn[:, 1]
        try: return acc, float(roc_auc_score(y1, sc)) if len(np.unique(y1)) > 1 else (acc, float('nan'))
        except: return acc, float('nan')
    pc = []
    for c in range(nc):
        yb = (y1 == c).astype(int)
        if 0 < yb.sum() < len(yb):
            try: pc.append(roc_auc_score(yb, yn[:, c]))
            except: pass
    return acc, float(np.mean(pc)) if pc else float('nan')

def ece(yt, ys, nb=15):
    y1 = np.asarray(yt).squeeze().astype(int)
    ys2 = np.nan_to_num(np.asarray(ys, dtype=np.float64))
    if ys2.ndim == 1: ys2 = np.stack([1 - ys2, ys2], 1)
    conf = ys2.max(1); pred = ys2.argmax(1); correct = (pred == y1).astype(float)
    bins = np.linspace(0, 1, nb + 1); e = 0.0
    for i in range(nb):
        m = (conf > bins[i]) & (conf <= bins[i + 1])
        if m.sum() > 0: e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(e)

def load_med(flag, sz=IMG_SIZE):
    info = INFO[flag]; DC = getattr(medmnist, info['python_class'])
    ic = info['n_channels']; task = info['task']
    nc = len(info['label']) if isinstance(info['label'], dict) else int(info['label'])
    root = os.path.join(OUTPUT_DIR, 'medmnist_data'); os.makedirs(root, exist_ok=True)
    ssz = sz if sz in (28, 64, 128, 224) else 224
    tr = DC(split='train', transform=build_tf(sz, ic, True), download=True, size=ssz, root=root)
    va = DC(split='val', transform=build_tf(sz, ic, False), download=True, size=ssz, root=root)
    te = DC(split='test', transform=build_tf(sz, ic, False), download=True, size=ssz, root=root)
    return tr, va, te, task, nc, ic

def load_cifar(n, sz=IMG_SIZE):
    root = os.path.join(OUTPUT_DIR, 'cifar'); os.makedirs(root, exist_ok=True)
    full = torchvision.datasets.CIFAR100(root, True, download=True, transform=build_tf(sz, 3, True))
    te = torchvision.datasets.CIFAR100(root, False, download=True, transform=build_tf(sz, 3, False))
    idx = torch.randperm(len(full), generator=torch.Generator().manual_seed(0))[:n].tolist()
    return Subset(full, idx), te, te, 'multi-class', 100, 3

print('Data + metrics ready')

Data + metrics ready


## 4. Training driver

In [8]:
def train_eval(model_kw, tr, va, te, task, nc, ic, epochs, bs, lr, seed):
    # model_kw: dict of kwargs for PAPCViT (use_pc, adaptive, progressive, etc.)
    torch.manual_seed(seed); np.random.seed(seed)
    if device.type == 'cuda': torch.cuda.manual_seed_all(seed)
    is_ml = task == 'multi-label, binary-class'

    model = PAPCViT(MODEL_NAME, IMG_SIZE, nc, ic, pretrained=True, **model_kw).to(device)
    use_pc = model_kw.get('use_pc', True)
    nw = min(10, os.cpu_count() or 4)
    kw = dict(num_workers=nw, pin_memory=True, persistent_workers=nw > 0)
    tl = DataLoader(tr, bs, shuffle=True, drop_last=True, **kw)
    el = DataLoader(te, bs * 2, **kw)

    # FIX: use keyword arguments so each value lands in the right slot.
    # The original positional call put 0.1 into cutmix_minmax (must be None or
    # a 2-element list), nc into prob (must be a float), 0.5 into mode (must be
    # a string), and 'batch' into correct_lam (must be a bool).
    mfn = Mixup(
        mixup_alpha=0.2,
        cutmix_alpha=1.0,
        prob=1.0,
        switch_prob=0.5,
        mode='batch',
        label_smoothing=0.1,
        num_classes=nc,
    ) if (not is_ml and nc > 2) else None

    opt = torch.optim.AdamW(model.param_groups(lr), weight_decay=0.05, betas=(0.9, 0.95))
    Tt = len(tl) * epochs; W = max(1, int(Tt * 0.05))
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (
        s / W if s < W else 0.5 * (1 + math.cos(math.pi * (s - W) / max(1, Tt - W)))))
    ema = EMA(model); t0 = time.time(); gs = 0

    for ep in range(epochs):
        model.train()
        for x, y in tl:
            model.set_progress(gs, Tt)
            x = x.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
            yl = y.long().squeeze(-1) if y.ndim > 1 else y.long()
            if mfn and not is_ml: x, tgt = mfn(x, yl)
            elif is_ml: tgt = y.float()
            else:
                tgt = F.one_hot(yl, nc).float()
                tgt = tgt * 0.9 + 0.1 / nc
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE, enabled=device.type == 'cuda'):
                lo, pl = model(x)
                main = (F.binary_cross_entropy_with_logits(lo, tgt) if is_ml
                        else -(tgt * F.log_softmax(lo, -1)).sum(-1).mean())
                loss = main + pl
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sch.step(); ema.update(model); gs += 1

    bk = ema.apply(model); model.eval()
    ys_all, ss_all = [], []
    with torch.no_grad():
        for x, y in el:
            x = x.to(device, non_blocking=True)
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE, enabled=device.type == 'cuda'):
                la, _ = model(x); lb, _ = model(torch.flip(x, [-1]))
            s = (torch.sigmoid if is_ml else lambda z: F.softmax(z, -1))((la + lb) / 2)
            ys_all.append(y.cpu().numpy()); ss_all.append(s.float().cpu().numpy())
    ema.restore(model, bk)
    ya = np.concatenate(ys_all); sa = np.concatenate(ss_all)
    acc, auc = compute_metrics(ya, sa, task, nc)
    ecv = ece(ya, sa)
    gv = float(np.mean(np.abs(model.gate_values()))) if use_pc else 0.0
    lw = model.learned_weights()

    del model, ema, opt, sch, tl, el
    if device.type == 'cuda': torch.cuda.empty_cache(); gc.collect()
    return dict(acc=acc, auc=auc, ece=ecv, gate=gv, lw=lw, t_min=(time.time() - t0) / 60)

print('train_eval ready')

train_eval ready


In [5]:
def train_eval(model_kw, tr, va, te, task, nc, ic, epochs, bs, lr, seed):
    # model_kw: dict of kwargs for PAPCViT (use_pc, adaptive, progressive, etc.)
    torch.manual_seed(seed); np.random.seed(seed)
    if device.type == 'cuda': torch.cuda.manual_seed_all(seed)
    is_ml = task == 'multi-label, binary-class'

    model = PAPCViT(MODEL_NAME, IMG_SIZE, nc, ic, pretrained=True, **model_kw).to(device)
    use_pc = model_kw.get('use_pc', True)
    nw = min(10, os.cpu_count() or 4)
    kw = dict(num_workers=nw, pin_memory=True, persistent_workers=nw > 0)
    tl = DataLoader(tr, bs, shuffle=True, drop_last=True, **kw)
    el = DataLoader(te, bs * 2, **kw)

    mfn = Mixup(0.2, 1.0, 0.1, nc, 1.0, 0.5, 'batch') if (not is_ml and nc > 2) else None
    opt = torch.optim.AdamW(model.param_groups(lr), weight_decay=0.05, betas=(0.9, 0.95))
    Tt = len(tl) * epochs; W = max(1, int(Tt * 0.05))
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (
        s / W if s < W else 0.5 * (1 + math.cos(math.pi * (s - W) / max(1, Tt - W)))))
    ema = EMA(model); t0 = time.time(); gs = 0

    for ep in range(epochs):
        model.train()
        for x, y in tl:
            model.set_progress(gs, Tt)
            x = x.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
            yl = y.long().squeeze(-1) if y.ndim > 1 else y.long()
            if mfn and not is_ml: x, tgt = mfn(x, yl)
            elif is_ml: tgt = y.float()
            else:
                tgt = F.one_hot(yl, nc).float()
                tgt = tgt * 0.9 + 0.1 / nc
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE, enabled=device.type == 'cuda'):
                lo, pl = model(x)
                main = (F.binary_cross_entropy_with_logits(lo, tgt) if is_ml
                        else -(tgt * F.log_softmax(lo, -1)).sum(-1).mean())
                loss = main + pl
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sch.step(); ema.update(model); gs += 1

    bk = ema.apply(model); model.eval()
    ys_all, ss_all = [], []
    with torch.no_grad():
        for x, y in el:
            x = x.to(device, non_blocking=True)
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE, enabled=device.type == 'cuda'):
                la, _ = model(x); lb, _ = model(torch.flip(x, [-1]))
            s = (torch.sigmoid if is_ml else lambda z: F.softmax(z, -1))((la + lb) / 2)
            ys_all.append(y.cpu().numpy()); ss_all.append(s.float().cpu().numpy())
    ema.restore(model, bk)
    ya = np.concatenate(ys_all); sa = np.concatenate(ss_all)
    acc, auc = compute_metrics(ya, sa, task, nc)
    ecv = ece(ya, sa)
    gv = float(np.mean(np.abs(model.gate_values()))) if use_pc else 0.0
    lw = model.learned_weights()

    del model, ema, opt, sch, tl, el
    if device.type == 'cuda': torch.cuda.empty_cache(); gc.collect()
    return dict(acc=acc, auc=auc, ece=ecv, gate=gv, lw=lw, t_min=(time.time() - t0) / 60)

print('train_eval ready')

train_eval ready


## 5. Configs

In [6]:
# Per-dataset: epochs/bs calibrated for ViT-Base on A100 40GB
CFG = {
    'pathmnist':      dict(ep=7,  bs=128, lr=1.5e-4),
    'bloodmnist':     dict(ep=18, bs=128, lr=1.5e-4),
    'dermamnist':     dict(ep=22, bs=96,  lr=1e-4),
    'breastmnist':    dict(ep=35, bs=32,  lr=5e-5),
    'pneumoniamnist': dict(ep=22, bs=96,  lr=1e-4),
    'retinamnist':    dict(ep=35, bs=32,  lr=5e-5),
    'organamnist':    dict(ep=10, bs=128, lr=1e-4),
    'organcmnist':    dict(ep=12, bs=128, lr=1e-4),
}

# Published baselines (AUC, ACC) for comparison table
PUB = {
    'pathmnist':      {'MedMamba-B':(0.999,0.964),'MedVit-S':(0.993,0.942)},
    'bloodmnist':     {'MedMamba-S':(0.999,0.984),'MedVit-S':(0.997,0.951)},
    'dermamnist':     {'MedVit-S':(0.937,0.780),'MedMamba-B':(0.925,0.757)},
    'pneumoniamnist': {'MedVit-S':(0.995,0.961),'MedMamba-S':(0.976,0.936)},
    'retinamnist':    {'MedMamba-X':(0.719,0.570),'MedVit-S':(0.773,0.561)},
    'breastmnist':    {'MedVit-S':(0.938,0.897),'MedMamba-B':(0.849,0.891)},
    'organamnist':    {'MedMamba-B':(0.998,0.953),'MedVit-S':(0.997,0.944)},
    'organcmnist':    {'MedMamba-S':(0.997,0.925),'MedVit-S':(0.995,0.917)},
}

# Model configs used across sessions
def mk_vanilla(): return dict(use_pc=False)
def mk_fixed(w=0.05): return dict(use_pc=True, pred_loss_weight=w)
def mk_prog(w=0.005): return dict(use_pc=True, pred_loss_weight=w, progressive=True)
def mk_adaptive(): return dict(use_pc=True, adaptive=True, w_max=0.01, w_l2=1e-3)
def mk_papc(): return dict(use_pc=True, adaptive=True, progressive=True, w_max=0.01, w_l2=1e-3)
def mk_clamped(w=0.05): return dict(use_pc=True, pred_loss_weight=w, clamp_gate=True)

RES = {}; T0 = time.time()
def out_json(): return os.path.join(OUTPUT_DIR, f'papc_{SESSION}_results.json')
def save():
    json.dump(RES, open(out_json(), 'w'), indent=2,
              default=lambda o: float(o) if hasattr(o, 'item') else str(o))
def tl(): return TIME_BUDGET_SEC - (time.time() - T0)  # time left

def run_grid(tag, datasets, conditions, seeds=3, save_every=True):
    # conditions: list of (name, model_kw_dict)
    for flag, cfg in datasets.items():
        if tl() < 600: print(f'TIME: skipping {flag}'); break
        try: tr, va, te, task, nc, ic = load_med(flag)
        except Exception as e: print(f'  {flag}: SKIP ({e})'); continue
        print(f'\n=== {flag} (n={len(tr)}) ===')
        RES.setdefault(tag, {}).setdefault(flag, {'n': len(tr)})
        for cname, mkw in conditions:
            RES[tag][flag].setdefault(cname, [])
            existing = len(RES[tag][flag][cname])
            for s in range(existing, seeds):
                if tl() < 300: break
                r = train_eval(mkw, tr, va, te, task, nc, ic, cfg['ep'], cfg['bs'], cfg['lr'], s)
                RES[tag][flag][cname].append(r)
                print(f'  {cname:14s} s{s}: AUC={r["auc"]:.4f} ACC={r["acc"]:.4f} '
                      f'ECE={r["ece"]:.4f} gate={r["gate"]:.3f} ({r["t_min"]:.1f}m)')
            if save_every: save()
        # print summary for this dataset
        for cname, _ in conditions:
            aucs = [r['auc'] for r in RES[tag][flag].get(cname, [])]
            if aucs:
                bp = PUB.get(flag, {})
                bst = max((v[0] for v in bp.values()), default=float('nan'))
                print(f'  {cname:14s} mean AUC={np.mean(aucs):.4f}+-{np.std(aucs):.4f}'
                      f'  (best pub={bst:.3f})')

print('Config ready | SESSION =', SESSION)

Config ready | SESSION = A


## 6A. Session A — Main SOTA table

In [9]:
if SESSION == 'A':
    print('SESSION A: Main SOTA table with ViT-Base')
    print('Conditions: Vanilla-B, HP-ViT-B (w=0.05, the failure), PAPC-B (novel)')
    run_grid('sota', CFG, [
        ('vanilla', mk_vanilla()),
        ('hp_0.05', mk_fixed(0.05)),
        ('PAPC',    mk_papc()),
    ], seeds=3)
    # Summary table
    print('\n' + '=' * 80)
    print('SOTA TABLE (Session A)')
    print(f'{"Dataset":16s} {"n":>6s}  {"Vanilla":>8s}  {"HP 0.05":>8s}  {"PAPC":>8s}  {"best pub":>8s}  {"PAPC>pub":>8s}')
    for flag in CFG:
        d = RES.get('sota', {}).get(flag, {})
        va = np.mean([r['auc'] for r in d.get('vanilla', [])]) if d.get('vanilla') else float('nan')
        hp = np.mean([r['auc'] for r in d.get('hp_0.05', [])]) if d.get('hp_0.05') else float('nan')
        pa = np.mean([r['auc'] for r in d.get('PAPC', [])]) if d.get('PAPC') else float('nan')
        bp = max((v[0] for v in PUB.get(flag, {}).values()), default=float('nan'))
        win = 'WIN' if pa > bp else ''
        print(f'  {flag:16s} {d.get("n","?"):>6}  {va:>8.4f}  {hp:>8.4f}  {pa:>8.4f}  {bp:>8.3f}  {win:>8s}')
    save()
    print(f'\nSession A done: {(time.time()-T0)/60:.1f} min')

SESSION A: Main SOTA table with ViT-Base
Conditions: Vanilla-B, HP-ViT-B (w=0.05, the failure), PAPC-B (novel)



=== pathmnist (n=89996) ===


  vanilla        s0: AUC=0.9981 ACC=0.9600 ECE=0.0603 gate=0.000 (10.9m)
  vanilla        s1: AUC=0.9971 ACC=0.9561 ECE=0.0597 gate=0.000 (10.9m)
  vanilla        s2: AUC=0.9977 ACC=0.9553 ECE=0.0599 gate=0.000 (10.9m)
  hp_0.05        s0: AUC=0.9908 ACC=0.9464 ECE=0.1048 gate=0.178 (38.6m)
  hp_0.05        s1: AUC=0.9909 ACC=0.9425 ECE=0.0321 gate=0.198 (38.6m)
  hp_0.05        s2: AUC=0.9927 ACC=0.9501 ECE=0.0736 gate=0.182 (38.6m)
  PAPC           s0: AUC=0.9974 ACC=0.9561 ECE=0.0708 gate=0.007 (38.6m)
  PAPC           s1: AUC=0.9980 ACC=0.9543 ECE=0.0752 gate=0.006 (38.6m)
  PAPC           s2: AUC=0.9975 ACC=0.9564 ECE=0.0571 gate=0.006 (38.6m)
  vanilla        mean AUC=0.9976+-0.0004  (best pub=0.999)
  hp_0.05        mean AUC=0.9915+-0.0009  (best pub=0.999)
  PAPC           mean AUC=0.9976+-0.0003  (best pub=0.999)


100%|██████████| 1.54G/1.54G [01:18<00:00, 19.6MB/s] 



=== bloodmnist (n=11959) ===
  vanilla        s0: AUC=0.9992 ACC=0.9901 ECE=0.0930 gate=0.000 (3.9m)
  vanilla        s1: AUC=0.9994 ACC=0.9895 ECE=0.0782 gate=0.000 (3.9m)
  vanilla        s2: AUC=0.9994 ACC=0.9904 ECE=0.0575 gate=0.000 (3.9m)
  hp_0.05        s0: AUC=0.9990 ACC=0.9661 ECE=0.1006 gate=0.100 (13.3m)
  hp_0.05        s1: AUC=0.9992 ACC=0.9813 ECE=0.0906 gate=0.101 (13.4m)
  hp_0.05        s2: AUC=0.9992 ACC=0.9810 ECE=0.1570 gate=0.084 (13.3m)
  PAPC           s0: AUC=0.9994 ACC=0.9871 ECE=0.0748 gate=0.015 (13.4m)
  PAPC           s1: AUC=0.9994 ACC=0.9880 ECE=0.0715 gate=0.017 (13.4m)
  PAPC           s2: AUC=0.9994 ACC=0.9889 ECE=0.0620 gate=0.011 (13.4m)
  vanilla        mean AUC=0.9993+-0.0001  (best pub=0.999)
  hp_0.05        mean AUC=0.9991+-0.0001  (best pub=0.999)
  PAPC           mean AUC=0.9994+-0.0000  (best pub=0.999)


...


=== dermamnist (n=7007) ===
  vanilla        s0: AUC=0.9821 ACC=0.8778 ECE=0.0720 gate=0.000 (3.0m)
  vanilla        s1: AUC=0.9812 ACC=0.8733 ECE=0.0765 gate=0.000 (3.0m)
  vanilla        s2: AUC=0.9822 ACC=0.8698 ECE=0.0948 gate=0.000 (3.0m)
  hp_0.05        s0: AUC=0.9751 ACC=0.8090 ECE=0.0645 gate=0.069 (9.7m)
  hp_0.05        s1: AUC=0.9569 ACC=0.3930 ECE=0.2385 gate=0.055 (9.7m)
  hp_0.05        s2: AUC=0.9591 ACC=0.4549 ECE=0.2429 gate=0.065 (9.7m)
  PAPC           s0: AUC=0.9811 ACC=0.8653 ECE=0.0886 gate=0.007 (9.8m)
  PAPC           s1: AUC=0.9809 ACC=0.8673 ECE=0.1028 gate=0.006 (9.8m)
  PAPC           s2: AUC=0.9799 ACC=0.8708 ECE=0.0776 gate=0.008 (9.7m)
  vanilla        mean AUC=0.9818+-0.0004  (best pub=0.937)
  hp_0.05        mean AUC=0.9637+-0.0081  (best pub=0.937)
  PAPC           mean AUC=0.9806+-0.0005  (best pub=0.937)


100%|██████████| 30.9M/30.9M [00:02<00:00, 15.0MB/s]



=== breastmnist (n=546) ===
  vanilla        s0: AUC=0.8159 ACC=0.6923 ECE=0.0849 gate=0.000 (0.5m)
  vanilla        s1: AUC=0.8140 ACC=0.6346 ECE=0.1073 gate=0.000 (0.5m)
  vanilla        s2: AUC=0.8227 ACC=0.8013 ECE=0.0603 gate=0.000 (0.5m)
  hp_0.05        s0: AUC=0.8808 ACC=0.7756 ECE=0.1628 gate=0.019 (1.4m)
  hp_0.05        s1: AUC=0.8390 ACC=0.8205 ECE=0.0576 gate=0.016 (1.4m)
  hp_0.05        s2: AUC=0.8585 ACC=0.7756 ECE=0.0679 gate=0.018 (1.4m)
  PAPC           s0: AUC=0.8632 ACC=0.7756 ECE=0.1578 gate=0.004 (1.4m)
  PAPC           s1: AUC=0.8194 ACC=0.7628 ECE=0.0696 gate=0.004 (1.4m)
  PAPC           s2: AUC=0.8119 ACC=0.6538 ECE=0.0824 gate=0.005 (1.4m)
  vanilla        mean AUC=0.8175+-0.0037  (best pub=0.938)
  hp_0.05        mean AUC=0.8594+-0.0171  (best pub=0.938)
  PAPC           mean AUC=0.8315+-0.0226  (best pub=0.938)


100%|██████████| 214M/214M [00:13<00:00, 15.6MB/s] 



=== pneumoniamnist (n=4708) ===
  vanilla        s0: AUC=0.9923 ACC=0.8830 ECE=0.0394 gate=0.000 (1.9m)
  vanilla        s1: AUC=0.9908 ACC=0.9583 ECE=0.0828 gate=0.000 (1.9m)
  vanilla        s2: AUC=0.9925 ACC=0.9423 ECE=0.0453 gate=0.000 (1.9m)
  hp_0.05        s0: AUC=0.9915 ACC=0.9407 ECE=0.0124 gate=0.058 (6.5m)
  hp_0.05        s1: AUC=0.9878 ACC=0.9263 ECE=0.0168 gate=0.064 (6.5m)
  hp_0.05        s2: AUC=0.9883 ACC=0.9119 ECE=0.0254 gate=0.060 (6.5m)
  PAPC           s0: AUC=0.9914 ACC=0.8638 ECE=0.0517 gate=0.005 (6.5m)
  PAPC           s1: AUC=0.9908 ACC=0.8958 ECE=0.0496 gate=0.003 (6.5m)
  PAPC           s2: AUC=0.9900 ACC=0.9391 ECE=0.0385 gate=0.007 (6.5m)
  vanilla        mean AUC=0.9919+-0.0008  (best pub=0.995)
  hp_0.05        mean AUC=0.9892+-0.0017  (best pub=0.995)
  PAPC           mean AUC=0.9907+-0.0006  (best pub=0.995)
TIME: skipping retinamnist

SOTA TABLE (Session A)
Dataset               n   Vanilla   HP 0.05      PAPC  best pub  PAPC>pub
  pathmnist      

## 7. Final

In [10]:
save()
print(f'\nSaved -> {out_json()}')
print(f'Total wall-clock: {(time.time()-T0)/60:.1f} min')
print(f'Download this JSON. Send it to me to fill in all paper tables and figures.')


Saved -> /teamspace/studios/this_studio/papc_A_results.json
Total wall-clock: 502.1 min
Download this JSON. Send it to me to fill in all paper tables and figures.
